#### DT-195

In [0]:
for env in ('p','qa'):
    try:
        # Check if the target is a view
        result = spark.sql(f"""
            SELECT table_type
            FROM system.information_schema.tables
            WHERE table_catalog = '{env}-cdh'
              AND table_schema = 'derivativearchive'
              AND table_name = 'dpaymentschedule'
        """)
        table_type = result.collect()[0]['table_type'] if result.count() > 0 else None
        if table_type == 'VIEW':
            raise Exception(f"`{env}-cdh`.derivativearchive.dpaymentschedule is a VIEW")
    except Exception as e:
        print(f"Skipping `{env}-cdh`.derivativearchive.dpaymentschedule: {e}")

    try:
        spark.sql(f'''
            update `{env}-cdh`.derivativearchive.dpaymentschedule
            set InstallmentStatus = 'Active'
            where InstallmentStatusCode = '0'
            and sourcesystem = 'qca'
            and InstallmentStatus <> 'Active';
        ''')
    except Exception as e:
        print(f"Failed to update InstallmentStatus to 'Active' for `{env}-cdh`.derivativearchive.dpaymentschedule: {e}")

    try:
        spark.sql(f'''
            update `{env}-cdh`.derivativearchive.dpaymentschedule
            set InstallmentStatus = 'Overdue'
            where InstallmentStatusCode = '1'
            and sourcesystem = 'qca'
            and InstallmentStatus <> 'Overdue';
        ''')
    except Exception as e:
        print(f"Failed to update InstallmentStatus to 'Overdue' for `{env}-cdh`.derivativearchive.dpaymentschedule: {e}")

    try:
        spark.sql(f'''
            update `{env}-cdh`.derivativearchive.dpaymentschedule
            set InstallmentStatus = 'Matched'
            where InstallmentStatusCode = '2'
            and sourcesystem = 'qca'
            and InstallmentStatus <> 'Matched';
        ''')
    except Exception as e:
        print(f"Failed to update InstallmentStatus to 'Matched' for `{env}-cdh`.derivativearchive.dpaymentschedule: {e}")

#### ITL-120

In [0]:
%sql
ALTER TABLE `p-cdh`.`primaryactive`.portfolio
ADD COLUMNS (Arotron_dec_flag INT AFTER Datadate);

In [0]:
%sql
UPDATE `p-cdh`.`primaryactive`.portfolio
SET Arotron_dec_flag = 0
WHERE Arotron_dec_flag IS NULL;

In [0]:
%sql
create or replace view `p-cdh`.`primary`.portfolio as 
select * from `p-cdh`.`primaryactive`.portfolio

#### DT-181

In [0]:
%sql

/*delete of duplicated records. Info from different systems*/
with realestatebuilding as (
  select distinct 
    try_cast(RealEstateKey as integer) RealEstateKey,
    CASE WHEN RealEstateStatus = 'Ενεργό' THEN 'Active'
      WHEN RealEstateStatus = 'Ανενεργό' THEN 'Inactive' else null end RealEstateStatus,
    BusinessDate,
    case when sourcesystem = 'arotron' then 'qca' else sourcesystem end as sourcesystemmapping
  FROM  primary.realestatebuilding
  WHERE 1=1
)

/*delete wrong records. info from other systems that shouldn't exist*/
delete from  derivativeactive.dcollateral a
where not exists(
  select * from realestatebuilding
  where 1=1
  and a.BusinessDate = realestatebuilding.BusinessDate
  and try_cast(a.RealEstateKey as integer)=try_cast(realestatebuilding.RealEstateKey  as integer)
  and ifnull(a.RealEstateStatus,'') = ifnull(realestatebuilding.RealEstateStatus,'')
  and a.SourceSystem = realestatebuilding.sourcesystemmapping
  )
  and a.collateralType='RE'
;

delete from  derivativeactive.dcollateral  a
where a.sourceSystem <> 'thesis'
and (
  a.portfolioName is null 
  or 
  exists (
    select distinct PortfolioName
    from primary.portfolio
    where 1=1
    and portfolio.unit_code_profits is not null
    and portfolio.businessdate = a.businessdate
    and portfolio.PortfolioName = a.PortfolioName
  )
)

###Refresh views

In [0]:
views_df = spark.sql("""
    SELECT table_name, table_schema, table_catalog
    FROM system.information_schema.tables
    WHERE table_catalog in ('p-cdh')
      AND table_schema in ('derivative', 'primary')
      and table_type='VIEW'
""")

# display(views_df)

for view_row in views_df.collect():
    table_name = view_row["table_name"]
    table_schema = view_row["table_schema"]
    table_catalog = view_row["table_catalog"]

    # print(table_name)

    full_view_name =f"`{table_catalog}`.{table_schema}.{table_name}"

    describe_full_view_name = spark.sql(f"""
              DESCRIBE EXTENDED {full_view_name}""")
    
    describe_full_view_name.createOrReplaceTempView("describe_full_view_name")

   # print(describe_full_view_name)

    describe_query = spark.sql(f"""
              select data_type--REGEXP_REPLACE(TRIM(data_type), '[^a-zA-Z0-9*<> ]', '') 
              AS cleaned_view_text 
              from describe_full_view_name 
              where col_name ='View Text'
              """)
    
    sql_view_creation = describe_query.collect()[0][0]

    create_or_replace_sql = f"""
          create or replace view {full_view_name} as 
          {sql_view_creation}
          """
    
    spark.sql(create_or_replace_sql)



In [0]:
views_df = spark.sql("""
    SELECT table_name, table_schema, table_catalog
    FROM system.information_schema.tables
    WHERE table_catalog in ('qa-cdh')
      AND table_schema in ('derivativearchive','primaryarchive')
      AND table_type='VIEW'
""")

for view_row in views_df.collect():
    table_name = view_row["table_name"]
    table_schema = view_row["table_schema"]
    table_catalog = view_row["table_catalog"]

    full_view_name = f"`{table_catalog}`.{table_schema}.{table_name}"

    try:
        describe_full_view_name = spark.sql(f"""
            DESCRIBE EXTENDED {full_view_name}
        """)

        describe_full_view_name.createOrReplaceTempView("describe_full_view_name")

        describe_query = spark.sql("""
            SELECT data_type
            FROM describe_full_view_name 
            WHERE col_name = 'View Text'
        """)

        sql_view_creation = describe_query.collect()[0][0]

        create_or_replace_sql = f"""
            CREATE OR REPLACE VIEW {full_view_name} AS 
            {sql_view_creation}
        """

        spark.sql(create_or_replace_sql)

        print(f"Successfully refreshed view: {full_view_name}")

    except Exception as e:
        print(f"   Failed to refresh view: {full_view_name}")


In [0]:
views_df = spark.sql("""
    SELECT table_name, table_schema, table_catalog
    FROM system.information_schema.tables
    WHERE table_catalog in ('qa-cdh')
      AND table_schema in ('derivativeactive','primaryactive')
      AND table_type='VIEW'
""")

for view_row in views_df.collect():
    table_name = view_row["table_name"]
    table_schema = view_row["table_schema"]
    table_catalog = view_row["table_catalog"]

    full_view_name = f"`{table_catalog}`.{table_schema}.{table_name}"

    try:
        describe_full_view_name = spark.sql(f"""
            DESCRIBE EXTENDED {full_view_name}
        """)

        describe_full_view_name.createOrReplaceTempView("describe_full_view_name")

        describe_query = spark.sql("""
            SELECT data_type
            FROM describe_full_view_name 
            WHERE col_name = 'View Text'
        """)

        sql_view_creation = describe_query.collect()[0][0]

        create_or_replace_sql = f"""
            CREATE OR REPLACE VIEW {full_view_name} AS 
            {sql_view_creation}
        """

        spark.sql(create_or_replace_sql)

        print(f"Successfully refreshed view: {full_view_name}")

    except Exception as e:
        print(f"   Failed to refresh view: {full_view_name}")


###Closed loans function update

In [0]:
env='p'
# Get the schema of the dexposure table
dexposure_table = spark.table(f"`{env}-cdh`.derivative.dexposure")
dexposure_schema_fields = dexposure_table.schema.fields


# Convert the schema fields into the required string format
dexposure_schema = ",\n".join([
    f"{field.name} {field.dataType.simpleString()}"
    for field in dexposure_schema_fields
])


spark.sql(f'''
CREATE OR REPLACE FUNCTION `{env}-cdh`.derivative.get_all_exposures_by_business_date(
    business_date Date
)
RETURNS TABLE 
( 
{dexposure_schema}
)
RETURN

SELECT B.*
FROM(
        SELECT  ExposureKey, MAX(BusinessDate) BusinessDate
        FROM `p-cdh`.derivative.dexposure
        WHERE BusinessDate <= business_date
        GROUP BY ExposureKey
) A
INNER JOIN   `{env}-cdh`.derivative.dexposure B
on a.ExposureKey = b.ExposureKey
and a.BusinessDate = b.BusinessDate
AND B.IsActiveFlagInt = 0
AND coalesce(B.JustificationCode,0) != 43023

UNION

SELECT * FROM `{env}-cdh`.derivative.dexposure
WHERE BusinessDate = business_date
AND IsActiveFlagInt = 1
''')